# 20261006 - Distributed-Memory Parallelism

![shared](../img/shared_memory_Gamma.png)

- OpenMP is shared memory computing
	- All the threads have access to the exact same pool of global memory
- Shared memory computing is generally quite limiting
	- Can only parallelize as much as a single machine (usually)
   

- Distributed memory allows for work to be distributed across multiple systems that don't have access to the exact same memory
- Any data which needs to be shared between processing elements must be **explicitly communicated over a network**
- Often written exclusively as SPMD

![distributed](../img/distributed_memory_Gamma.png)

## Architecture of Distributed Memory Systems
- The network is the defining aspect of a distributed memory system
- Can be characterized by it's (graph) topology
    - Graph created by nodes, which are either compute nodes or network switches
    - Edges of the graph are connections between compute and switches
   

- Messages on the network must "hop" node-to-node via each link
- Diameter describes the maximum (over each pair of compute nodes) least number of hops
    - e.g. Assuming optimal message routing, what's the worst case hop-count
- Total latency to send a message is dictated by the number of hops

### Network Topology

![network topology](../img/network_topology_aurora_dataflow.png)

*From [ALCF Presentation](https://www.alcf.anl.gov/sites/default/files/2024-07/Aurora-DataFlow-28Feb24.pdf)*

- Tree: nested bisection tree
    - Fat tree puts more bandwidth higher up the tree
- Torus: periodically connected mesh structure
- Dragonfly: Hierarchy of all-to-all connections

![aurora dragonfly](../img/aurora_dragonfly_network.png)

*From [*Aurora: Architecting Argonne's First Exascale Supercomputer for Accelerated Scientific Discovery*](https://doi.org/10.48550/arXiv.2509.08207)

## Network within compute node


- Network connections can also apply within a compute node

![aurora compute blade](../img/aurora_compute_blade_network.png)

*From [Aurora: Architecting Argonne's First Exascale Supercomputer for Accelerated Scientific Discovery](https://doi.org/10.48550/arXiv.2509.08207)*

- GPU Stacks (silicon dies on same package) are networked to each other
- GPUs are networked together
    - In an all-to-all fashion
- CPUs are networked to GPUs and themselves
- CPUS are networked to the actual Network Interface Cards (NICs)

Note that latencies are not shown, which may be the more important factor than bandwidth

### How does the network layout/topology impact the user/developer?

- Often don't need to be concerned with network topology
    - Communication interfaces (e.g. MPI) usually take them into account automatically
        - For example, GPU-aware MPI will use inter-GPU network links if possible
    - Sometimes you may want to do some tuning of the interfaces though for your specific application (outside class scope)

- Can impact how you decompose work across the machine
    - We may want attempt to reduce inter-node communication at the expense of intranode communication
   

- Most often, they are useful to explain performance changes
    - e.g. going from 1 GPU to a full node, or a full node to multiple nodes, etc.

## Performance Model

Up till now, we've had the general performance model

$$ T = \ell + W/R$$

where 
- $T$ is the time taken to do something
- $\ell$ is the latency
- $W$ is the amount of work to be done
- $R$ is the rate at which work is done

Previous, we've looked at memory and compute, where

| | $W$ | $R$ | 
| --- | --- | --- |
| Memory | Bytes transfered | MB/s |
| Compute | FLOPs performed | FLOP/s |

Distributed memory parallelism introduces a new addition: Networking
- Same $W$ and $R$ units (and $\ell$), but with very different range of values

- For memory and compute, we've primarily focused on the $W/R$ term
    - e.g. How do we optimize such that we're memory-bound or compute-bound (operating at peak $R$)
- It's easy to focus on that, as it operates with $W$, the actual work we want to do
- For shared memory, single-package machines, we deal with large enough $W$ and small enough $R$ that $\ell$ often isn't relevant
- Distributed memory parallelism will often bring latency to the forefront

 
| | $\approx \max(R)$ | $\approx \ell$ |
| --- | --- | --- |
| Compute | 1 TF/s | $> 1\text{ns}$ (1 clock cycle) |
| Memory | 300 GB/s | $ 100 \text{ns}$ (300 clock cycles) |
| Network | 200 GB/s | $ > 2 \mu\text{s}$ (600,000 clock cycles) |

### Latency isn't getting any better

![network latency table](../img/network_latency_timeline_Jed.png)

- Latency has plataued (hasn't gotten better since 2015 either)
- Bandwidth has improved (could go even faster, but not worth it in most cases)

### What does this mean?
1. We should reduce the total amount of communications necessary (*not* necessarily require less data communicated)
    - Choose algorithms that are *communication avoiding*
    - Remove any unnecessary communications, particularly in "hot loops"
   

2. We should "hide" the latency if we can
    - Overlap communication with other useful work
    - Computing while communicating
    - Batching communication together (where it makes sense)

---
- Latency particularly important for parallel syncronizations
    - A simple `omp_barrier()`-like construct will now take *at least* $2 \mu\text{s}$ to do
    - Often will take significantly longer, as not every process will reach the barrier at the same "time"
   

- Reminder: Strong scaling is very sensitive to latency
    - Reducing communication and hiding latency will dramatically improve strong scaling